In [ ]:
import openml
import sys
sys.path.append('benchmark_dim_red')
from methods import methods
import time
import lzma
import pickle
import os
import submitit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd
from sklearn.metrics.cluster import adjusted_rand_score, \
    adjusted_mutual_info_score, homogeneity_completeness_v_measure, fowlkes_mallows_score, \
        silhouette_score, calinski_harabasz_score
import plotly.express as px

In [ ]:
results = []
suite = openml.study.get_suite(99)
for task_id in suite.tasks:
    task = openml.tasks.get_task(task_id, download_splits=False)
    data, targets, categorical, attribute_names = task.get_dataset().get_data(target=task.target_name)
    if data.shape[1] <= 20:
        continue    
    for cv in range(1, 10):
        for method_name in methods.keys():
            for n_components in [5, 10, 20, 40]:
                if data.shape[1] <= n_components:
                    continue
                if data.shape[0] > 10_000:
                    continue
                try:
                    with lzma.open(f"/bench_dim_red_claix/new_bench_dim_red/benchmark_dim_red/ml_data_reduced/{task_id}_{cv}_{method_name}_{n_components}_cluster.pkl.xz", "rb") as file:
                        ari, ami, hcv, fms = pickle.load(file)

                        results.append([task_id, cv, method_name, n_components, ari, ami, hcv, fms])
                except FileNotFoundError:
                    print(f"Missing {task_id}_{cv}_{method_name}_{n_components}")


In [ ]:
df = pd.DataFrame(results)
df.columns = ["task_id", "cv", "method", "n_components", "ari", "ami", "hcv", "fms"]
df.to_parquet("/benchmark_dim_red/agg_res/ml/results_cluster.parquet")

In [ ]:
df = pd.DataFrame(results)
df.columns = ["task_id", "cv", "method", "n_components", "ari", "ami", "hcv", "fms"]
df.to_parquet("benchmark_dim_red/agg_res/ml/results_clusterv2.parquet")

In [ ]:
df = pd.read_parquet("benchmark_dim_red/agg_res/ml/results_cluster.parquet")
df = pd.concat([df.drop('hcv', axis=1), pd.DataFrame(df['hcv'].tolist(), columns=['homogeneity', 'completeness', 'v_measure'])], axis=1)


In [ ]:
for metric in ['ari', 'ami', 'fms', 'homogeneity', 'completeness', 'v_measure']:
    for n_components in [5, 10, 20, 40]:
        df_n = df[df['n_components'] == n_components]
        df_n = df_n.groupby(['task_id', 'method']).mean().reset_index()
        
        fig = px.box(df, x='method', y=metric, color='method',
                                color_discrete_map={
                                    'PCA_linear':px.colors.qualitative.Plotly[0],
                                    'PCA_rbf':px.colors.qualitative.Dark24[5],
                                    'PCA_sigmoid':px.colors.qualitative.Safe[10],
                                    'PCA_cosine':px.colors.qualitative.Safe[9],
                                    'PCA_poly': px.colors.qualitative.Safe[3],
                                    'IPCA':px.colors.qualitative.Safe[5],
                                    'FastICA':px.colors.qualitative.Plotly[1],
                                    'TruncatedSVD':px.colors.qualitative.Plotly[2],
                                    'lolP':px.colors.qualitative.Plotly[3],
                                    'isomap':px.colors.qualitative.Plotly[4],
                                    'lle':px.colors.qualitative.Plotly[5],
                                    'pls_reg':px.colors.qualitative.Plotly[6],
                                    'DictionaryLearning':px.colors.qualitative.Plotly[7],
                                    'sparsePCA':px.colors.qualitative.Plotly[8],
                                    'FA':px.colors.qualitative.Plotly[9],
                                    'BernoulliRBM':px.colors.qualitative.G10[4],
                                    'GaussianRandomProjection':px.colors.qualitative.G10[8],
                                    'SparseRandomProjection':px.colors.qualitative.G10[9],
                                })
        fig.update_xaxes(categoryorder='array', 
                            categoryarray=['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                        'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                        'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                        'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM'], title=None)
        fig.update_layout(
            xaxis = dict(
                tickmode = 'array',
                tickvals = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24],
                ticktext =['PCA_linear', 'PCA_rbf', 'PCA_sigmoid', 
                                        'PCA_cosine', 'PCA_poly', 'IPCA', 'FastICA', 
                                        'DictionaryLearning', 'sparsePCA', 'GaussianRandomProjection', 'SparseRandomProjection', 'lolP', 'TruncatedSVD',
                                        'isomap', 'lle', 'pls_reg', 'FA', 'BernoulliRBM']
            )
        )

        if metric == 'ari':
            metric_name = 'Adjusted Rand Index'
        elif metric == 'ami':
            metric_name = 'Adjusted Mutual Information'
        elif metric == 'fms':
            metric_name = 'Fowlkes-Mallows Score'
        elif metric == 'homogeneity':
            metric_name = 'Homogeneity'
        elif metric == 'completeness':
            metric_name = 'Completeness'
        elif metric == 'v_measure':
            metric_name = 'V-Measure'
        fig.update_layout(title_text=None,
                template='plotly_white',
                showlegend=False,
                legend_title=None,
                width=200, height=120, margin=dict(l=0, r=0, t=0, b=0),
                font=dict(
                    family="Times New Roman",
                    size=10,
                    color="Black"
                ),
                title_font=dict(
                    family="Times New Roman",
                    size=10,
                    color="Black"
                ),
                yaxis_title=None,
                )
        fig.update_xaxes(showticklabels=False)
        fig.write_image(f'benchmark_dim_red/figures/ml_cluster/{n_components}_{metric}.pdf')


In [ ]:
df

In [ ]:
dims = []
suite = openml.study.get_suite(99)
for task_id in suite.tasks:
    task = openml.tasks.get_task(task_id, download_splits=False)
    data, targets, categorical, attribute_names = task.get_dataset().get_data(target=task.target_name)
    if data.shape[1] <= 20:
        continue

    dims.append(data.shape[1])

In [ ]:
np.max(dims)